# EXAMEN BIMESTRAL
# Sistema de Recuperación de Información — Corpus SciFact (`BeIR/scifact`)

**Asignatura:** Recuperación de la Información  

**Nombre:** Luis Morocho
**Corpus asignado:** SciFact (`BeIR/scifact` y `BeIR/scifact-qrels`)  

---

## 1. Carga y Exploración del Corpus

* **Corpus:** 5,183 abstracts de artículos científicos biomédicos en inglés, identificados mediante su `_id` original (sin fragmentación del texto, respetando la unidad de evaluación).
* **Consultas:** Afirmaciones científicas (*claims*) utilizadas para consultar el sistema.
* **Juicios de relevancia (qrels - partición test):** 339 pares consulta-documento que establecen la relevancia real para evaluar el rendimiento de los modelos.

In [1]:
import nltk
from datasets import load_dataset

# Descargar lista de stopwords en inglés
nltk.download('stopwords', quiet=True)

# Cargar el corpus, las consultas y los juicios de relevancia (qrels)
print("Descargando SciFact desde Hugging Face...")
corpus_ds = load_dataset("BeIR/scifact", "corpus")["corpus"]
queries_ds = load_dataset("BeIR/scifact", "queries")["queries"]
qrels_ds = load_dataset("BeIR/scifact-qrels", split="test")

print(f"Documentos cargados: {len(corpus_ds)}")
print(f"Consultas cargadas: {len(queries_ds)}")
print(f"Qrels (test) cargados: {len(qrels_ds)}")

# Inspeccionar estructura de un ejemplo
print("\n--- Ejemplo de Documento ---")
print(corpus_ds[0])

print("\n--- Ejemplo de Consulta ---")
print(queries_ds[0])

print("\n--- Ejemplo de Qrel ---")
print(qrels_ds[0])

c:\Users\Luis\Desktop\SEP\RI\Examen Bimestral\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Descargando SciFact desde Hugging Face...


c:\Users\Luis\Desktop\SEP\RI\Examen Bimestral\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Luis\.cache\huggingface\hub\datasets--BeIR--scifact. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Luis\Desktop\SEP\RI\Examen Bimestral\.venv\Lib\site-packages\huggingface_hub\file_download.py:

Documentos cargados: 5183
Consultas cargadas: 1109
Qrels (test) cargados: 339

--- Ejemplo de Documento ---
{'_id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8

## 2. Preprocesamiento de Texto y Construcción del Índice Invertido

Para representar adecuadamente los documentos científicos en inglés y conservar la correspondencia exacta con cada identificador original (`_id`), se aplican las siguientes decisiones de preprocesamiento:

1. **Concatenación de campos:** Se une el título (title) y el cuerpo del abstract (text) para indexar la totalidad de la información semántica de cada documento.
2. **Normalización:** Conversión de todo el texto a minúsculas para evitar duplicidad de términos por capitalización.
3. **Tokenización alfanumérica (\b[a-z0-9]+\b):** Se extraen palabras y valores numéricos, eliminando signos de puntuación pero conservando números, dosis, porcentajes y códigos de genes relevantes en literatura biomédica.
4. **Eliminación de *stopwords* y tokens cortos:** Se filtran palabras vacías del idioma inglés (`nltk.corpus.stopwords`) y tokens de un solo carácter que no aportan valor discriminativo.
5. **Stemming (`PorterStemmer`):** Reducción morfológica de los términos a su raíz léxica para agrupar variantes flexivas (ej. *cells*, *cellular* $\rightarrow$ *cell*).


In [5]:
import re
from collections import defaultdict
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text: str) -> list[str]:
    """
    Pipeline de preprocesamiento para SciFact (literatura biomédica en inglés):
    1. Normalización: conversión a minúsculas.
    2. Tokenización: expresión regular alfanumérica para conservar números/dosis/genes.
    3. Filtrado: eliminación de stopwords en inglés y tokens de longitud <= 1.
    4. Stemming: reducción morfológica con PorterStemmer.
    """
    if not text:
        return []
    tokens = re.findall(r'\b[a-z0-9]+\b', text.lower())
    return [
        stemmer.stem(tok)
        for tok in tokens
        if tok not in stop_words and len(tok) > 1
    ]

# Estructuras conservando el identificador original (_id)
doc_ids = []
documents_meta = {}      # doc_id -> {"title": ..., "text": ...}
tokenized_corpus = []    # Lista de tokens por documento (para BM25)
processed_strings = []   # Strings limpios por documento (para TF-IDF)
inverted_index = defaultdict(lambda: defaultdict(int))  # término -> {doc_id: frecuencia}

for row in corpus_ds:
    d_id = str(row["_id"])
    title = row["title"] or ""
    text = row["text"] or ""
    full_text = f"{title} {text}".strip()
    
    tokens = preprocess_text(full_text)
    
    doc_ids.append(d_id)
    documents_meta[d_id] = {"title": title, "text": text}
    tokenized_corpus.append(tokens)
    processed_strings.append(" ".join(tokens))
    
    # Construcción del índice invertido
    for tok in tokens:
        inverted_index[tok][d_id] += 1

print(f"Documentos indexados: {len(doc_ids)}")
print(f"Tamaño del vocabulario único: {len(inverted_index)} términos")
termino_prueba = stemmer.stem("cancer")
print(f"Ejemplo índice invertido para '{termino_prueba}' (primeros 5 docs): {list(inverted_index[termino_prueba].items())[:5]}")

Documentos indexados: 5183
Tamaño del vocabulario único: 25978 términos
Ejemplo índice invertido para 'cancer' (primeros 5 docs): [('5836', 1), ('33370', 1), ('213017', 1), ('226488', 1), ('293661', 2)]


## 3. Modelos de Recuperación: TF-IDF (Similitud Coseno) y BM25



In [6]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

# 1. Construcción del modelo TF-IDF (usando el texto ya tokenizado y procesado)
tfidf_vectorizer = TfidfVectorizer(tokenizer=str.split, preprocessor=None, lowercase=False)
tfidf_matrix = tfidf_vectorizer.fit_transform(processed_strings)

# 2. Construcción del modelo BM25
bm25_model = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

def search_documents(query: str, model: str = "bm25", top_k: int = 10) -> list[dict]:
    """
    Recupera y ordena los top_k documentos más relevantes para una consulta
    utilizando 'bm25' o 'tfidf'.
    """
    query_tokens = preprocess_text(query)
    if not query_tokens:
        return []
        
    if model.lower() == "bm25":
        scores = bm25_model.get_scores(query_tokens)
    elif model.lower() == "tfidf":
        query_str = " ".join(query_tokens)
        query_vec = tfidf_vectorizer.transform([query_str])
        scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    else:
        raise ValueError("Modelo no soportado. Use 'bm25' o 'tfidf'.")
    
    # Ordenar índices de mayor a menor puntuación
    sorted_indices = np.argsort(scores)[::-1]
    
    results = []
    seen_ids = set()
    rank = 1
    
    for idx in sorted_indices:
        if len(results) >= top_k:
            break
        score = float(scores[idx])
        if score <= 0:
            continue
        d_id = doc_ids[idx]
        if d_id in seen_ids:
            continue
        seen_ids.add(d_id)
        
        meta = documents_meta[d_id]
        raw_text = meta["text"]
        snippet = raw_text[:220].replace("\n", " ") + ("..." if len(raw_text) > 220 else "")
        
        results.append({
            "rank": rank,
            "document_id": d_id,
            "score": round(score, 4),
            "title": meta["title"],
            "snippet": snippet
        })
        rank += 1
        
    return results

# Prueba comparativa rápida con una consulta de ejemplo
consulta_ejemplo = "0-dimensional biomaterials show inductive properties."
print(f"Consulta de prueba: '{consulta_ejemplo}'\n")

print("--- Top 3 resultados con BM25 ---")
for r in search_documents(consulta_ejemplo, model="bm25", top_k=3):
    print(f"#{r['rank']} [ID: {r['document_id']}] (Score: {r['score']}) -> {r['title']}")

print("\n--- Top 3 resultados con TF-IDF ---")
for r in search_documents(consulta_ejemplo, model="tfidf", top_k=3):
    print(f"#{r['rank']} [ID: {r['document_id']}] (Score: {r['score']}) -> {r['title']}")

c:\Users\Luis\Desktop\SEP\RI\Examen Bimestral\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Consulta de prueba: '0-dimensional biomaterials show inductive properties.'

--- Top 3 resultados con BM25 ---
#1 [ID: 21257564] (Score: 9.9258) -> How long will long-term potentiation last?
#2 [ID: 18953920] (Score: 9.7594) -> The Epithelial-Mesenchymal Transition Generates Cells with Properties of Stem Cells
#3 [ID: 40212412] (Score: 9.4518) -> Periosteal bone formation--a neglected determinant of bone strength.

--- Top 3 resultados con TF-IDF ---
#1 [ID: 10608397] (Score: 0.0998) -> High-performance neuroprosthetic control by an individual with tetraplegia.
#2 [ID: 31543713] (Score: 0.0968) -> Data analysis methods for detection of differential protein expression in two-dimensional gel electrophoresis.
#3 [ID: 42421723] (Score: 0.0944) -> Induction of galactokinase in Saccharomyces cerevisiae: kinetics of induction and glucose effects.


## 4. Evaluación de los Modelos de Recuperación (Partición `test`)

Para evaluar objetivamente el rendimiento de ambos modelos, se utilizan las consultas pertenecientes a la partición `test` de `BeIR/scifact-qrels` junto con sus respectivos juicios de relevancia (`qrels`). Se calculan tres métricas estándar de recuperación de información en el corte $k = 10$:

* **Precision@10:** Proporción de documentos relevantes entre los 10 primeros recuperados.
* **Recall@10:** Proporción del total de documentos relevantes existentes para la consulta que fueron recuperados en el top 10.
* **NDCG@10 (*Normalized Discounted Cumulative Gain*):** Evalúa la calidad del ordenamiento (*ranking*), premiando que los documentos relevantes aparezcan en las primeras posiciones.

In [7]:
import pandas as pd

# Mapear ID de consulta -> texto de la consulta
queries_dict = {str(row["_id"]): row["text"] for row in queries_ds}

# Agrupar juicios de relevancia (qrels) de test: query_id -> {doc_id: relevancia}
qrels_test = defaultdict(dict)
for row in qrels_ds:
    q_id = str(row["query-id"])
    d_id = str(row["corpus-id"])
    rel = int(row["score"])
    if rel > 0:
        qrels_test[q_id][d_id] = rel

def compute_metrics_at_k(retrieved_ids: list[str], relevant_dict: dict[str, int], k: int = 10):
    retrieved_k = retrieved_ids[:k]
    num_rel = len(relevant_dict)
    if num_rel == 0:
        return 0.0, 0.0, 0.0
    
    hits = [1 if d_id in relevant_dict else 0 for d_id in retrieved_k]
    
    precision = sum(hits) / k
    recall = sum(hits) / num_rel
    
    # DCG@K e IDCG@K para NDCG@K
    dcg = sum(
        relevant_dict.get(d_id, 0) / np.log2(idx + 2)
        for idx, d_id in enumerate(retrieved_k)
    )
    ideal_rels = sorted(relevant_dict.values(), reverse=True)[:k]
    idcg = sum(
        rel / np.log2(idx + 2)
        for idx, rel in enumerate(ideal_rels)
    )
    ndcg = dcg / idcg if idcg > 0 else 0.0
    
    return precision, recall, ndcg

def evaluate_model(model_name: str, k: int = 10):
    precisions, recalls, ndcgs = [], [], []
    evaluated_queries = []
    
    for q_id, rel_docs in qrels_test.items():
        if q_id not in queries_dict:
            continue
        q_text = queries_dict[q_id]
        evaluated_queries.append((q_id, q_text))
        
        res = search_documents(q_text, model=model_name, top_k=k)
        ret_ids = [r["document_id"] for r in res]
        
        p, r, ndcg = compute_metrics_at_k(ret_ids, rel_docs, k=k)
        precisions.append(p)
        recalls.append(r)
        ndcgs.append(ndcg)
        
    return {
        "Modelo": model_name.upper(),
        "Consultas Evaluadas": len(evaluated_queries),
        f"Precision@{k}": round(float(np.mean(precisions)), 4),
        f"Recall@{k}": round(float(np.mean(recalls)), 4),
        f"NDCG@{k}": round(float(np.mean(ndcgs)), 4)
    }, evaluated_queries

res_tfidf, eval_queries_list = evaluate_model("tfidf", k=10)
res_bm25, _ = evaluate_model("bm25", k=10)

df_comparacion = pd.DataFrame([res_tfidf, res_bm25])
print("=== TABLA COMPARATIVA DE MÉTRICAS EN TEST ===")
display(df_comparacion)

print("\nListado de las primeras 10 consultas de test evaluadas (para reproducibilidad):")
df_queries_sample = pd.DataFrame(eval_queries_list[:10], columns=["query_id", "query_text"])
display(df_queries_sample)

=== TABLA COMPARATIVA DE MÉTRICAS EN TEST ===


,Modelo,Consultas Evaluadas,Precision@10,Recall@10,NDCG@10
0,TFIDF,300,0.0820,0.7476,0.5915
1,BM25,300,0.0913,0.8295,0.6856



Listado de las primeras 10 consultas de test evaluadas (para reproducibilidad):


,query_id,query_text
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
5,42,A high microerythrocyte count raises vulnerabi...
6,48,"A total of 1,000 people in the UK are asymptom..."
7,49,ADAR1 binds to Dicer to cleave pre-miRNA.
8,50,AIRE is expressed in some skin tumors.
9,51,ALDH1 expression is associated with better bre...


### Análisis y Comparación de Resultados (TF-IDF vs. BM25)

Al evaluar los dos modelos con las **300 consultas** del conjunto `test` de **SciFact** (`BeIR/scifact-qrels`), se pueden observar tres puntos principales:

1. **Mejor rendimiento de BM25 sobre TF-IDF:**  
2. **¿Por qué Precision@10 es baja (~0.09) mientras Recall@10 es alta (~0.83)?**  
   A primera vista el 0.0913 de precisión parece bajo, pero tiene sentido por cómo está construido el dataset SciFact: casi todas las consultas tienen apenas 1 o 2 documentos relevantes en todo el corpus de 5,183 artículos. Si una consulta solo tiene 1 documento relevante y el modelo lo pone en primer lugar, el Recall@10 es perfecto (1.0), pero la Precision@10 nunca puede pasar de 1/10 = 0.10. Por eso, en este caso, fijarnos en el **Recall@10 (82.95%)** y el **NDCG@10 (0.6856)** nos dice mucho más y confirma que el sistema sí está encontrando los artículos correctos y poniéndolos arriba en el ranking.
3. **Importancia de conservar números y códigos en el preprocesamiento:**  
   Haber mantenido tokens alfanuméricos (\b[a-z0-9]+\b) en lugar de borrar todos los números ayudó bastante, ya que varias consultas científicas dependen de porcentajes, cantidades o nombres de genes/proteínas (como *"1,000 genomes project"*, *"5% of perinatal mortality"* o *"ADAR1"*).

---

## 5. Exportación de los Índices para el Servicio Web

Por último, guardamos en un archivo binario (`App/ir_index.pkl`) los índices ya calculados, el vocabulario, los metadatos de los documentos y los modelos TF-IDF y BM25. De esta forma, cuando levantemos la API web, no tendrá que volver a procesar los 5,183 documentos desde cero en cada consulta, sino que los cargará directo en memoria para responder rápido.

In [9]:
import os
import pickle

os.makedirs("App", exist_ok=True)

artifacts_payload = {
    "doc_ids": doc_ids,
    "documents_meta": documents_meta,
    "tfidf_vectorizer": tfidf_vectorizer,
    "tfidf_matrix": tfidf_matrix,
    "bm25_model": bm25_model
}

output_path = os.path.join("App", "ir_index.pkl")
with open(output_path, "wb") as f:
    pickle.dump(artifacts_payload, f)

size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Índice exportado exitosamente en '{output_path}' ({size_mb:.2f} MB)")

Índice exportado exitosamente en 'App\ir_index.pkl' (18.47 MB)
